# 01: position, velocity and particle tracking

The lecture demonstration shows how a position measurement also corrects velocity. E1 applies the same covariance coupling to particle tracking (25 minutes); E3 adapts it to timestamped drone sensors. Build the simulation/filter yourself or with AI assistance; a complete reference is supplied.

Use **Kernel > Change Kernel > Crazyflie lectures (.venv)** and run setup, then the following cells with Shift+Enter. Results go to `data/<run_id>/`; choose a new run label for a new session.


Continue afterwards with [02 calibration](02_calibration.ipynb), then [03 height and velocity](03_height_velocity.ipynb). Each notebook runs in its own kernel.


In [ ]:
from pathlib import Path
import json
import sys

# Shared helpers are beside day2; data stays beside these notebooks.
notebook_dir = Path.cwd()
if (notebook_dir / "code/day2").is_dir():
    notebook_dir = notebook_dir / "code/day2"
sys.path.insert(0, str(notebook_dir.resolve().parent))

import numpy as np
from IPython.display import Image, display
from day2.notebook_tools import next_attempt

run_id = "trial-01"  # Label this session; each comparison gets a fresh attempt folder.
output = notebook_dir / "data" / run_id
output.mkdir(parents=True, exist_ok=True)
print("Results:", output.resolve())


### Position measurements reveal velocity

Start with state $(x,v)$ and position observations only.
The velocity correction is $\Delta v=C_{vx}^{pred}r/(C_{xx}^{pred}+V)$.
Inspect the update; replace time/velocity with z/slope for E1.


In [ ]:
# Exact constant-speed motion: uncertainty comes from initial state and position noise.
demo_time = np.arange(21) * 0.2  # seconds
demo_true_velocity = 0.15  # m/s
demo_true_position = 0.2 + demo_true_velocity * demo_time
demo_position_std = 0.03  # metres
demo_V = demo_position_std**2
demo_rng = np.random.default_rng(7)
demo_measurements = demo_true_position + demo_rng.normal(0, demo_position_std, len(demo_time))

def position_velocity_update(predicted, predicted_C, measurement, V):
    H = np.array([1.0, 0.0])  # The sensor observes position.
    residual = measurement - H @ predicted
    R = H @ predicted_C @ H + V
    K = predicted_C @ H / R
    corrected = predicted + K * residual
    # Joseph form keeps the covariance symmetric and positive under roundoff.
    A = np.eye(2) - np.outer(K, H)
    corrected_C = A @ predicted_C @ A.T + np.outer(K, K) * V
    return corrected, corrected_C, K, residual

demo_state = np.zeros(2)  # Initially, position and velocity are unknown.
demo_C = np.diag([1.0, 1.0])  # Variances: m^2 and (m/s)^2.
demo_states, demo_covariances, demo_updates = [], [], []
for index, measurement in enumerate(demo_measurements):
    dt = 0.0 if index == 0 else demo_time[index] - demo_time[index - 1]
    F = np.array([[1.0, dt], [0.0, 1.0]])
    # Q=0 for this exact constant-velocity example.
    predicted = F @ demo_state
    predicted_C = F @ demo_C @ F.T
    # F creates C_vx: a velocity error also becomes a position error over dt.
    demo_state, demo_C, K, residual = position_velocity_update(
        predicted, predicted_C, measurement, demo_V)
    demo_updates.append(dict(predicted=predicted, C=predicted_C, K=K, residual=residual))
    demo_states.append(demo_state.copy())
    demo_covariances.append(demo_C.copy())
demo_states = np.asarray(demo_states)
demo_covariances = np.asarray(demo_covariances)

demo_update = demo_updates[1]  # First update after a nonzero time step.
demo_delta_v = demo_update["K"][1] * demo_update["residual"]
demo_without_cross_covariance = demo_update["C"].copy()
demo_without_cross_covariance[0, 1] = demo_without_cross_covariance[1, 0] = 0.0
demo_uncoupled_state, _, _, _ = position_velocity_update(
    demo_update["predicted"], demo_without_cross_covariance,
    demo_measurements[1], demo_V)
print(f"First observation (t=0): velocity gain = {demo_updates[0]['K'][1]:.3f} s^-1")
print(f"Next prediction: C_vx = {demo_update['C'][1, 0]:.4f} m^2/s")
print(f"Position residual = {demo_update['residual']:.4f} m; K_v = {demo_update['K'][1]:.4f} s^-1")
print(f"Velocity correction = K_v * residual = {demo_delta_v:.4f} m/s")
print(f"Same update with C_vx=0: velocity correction = "
      f"{demo_uncoupled_state[1] - demo_update['predicted'][1]:.4f} m/s")


In [ ]:
import matplotlib.pyplot as plt

demo_figure, demo_axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
demo_axes[0].plot(demo_time, demo_true_position, "k--", label="True position")
demo_axes[0].plot(demo_time, demo_measurements, ".", color="C1", label="Position measurements")
demo_axes[0].set_ylabel("Position [m]")
demo_axes[1].axhline(demo_true_velocity, color="black", linestyle="--", label="True velocity")
demo_axes[1].set_ylabel("Velocity [m/s]")
for component, axis in enumerate(demo_axes):
    sigma = np.sqrt(demo_covariances[:, component, component])
    axis.plot(demo_time, demo_states[:, component], color="C0", label="Filtered estimate")
    axis.fill_between(demo_time, demo_states[:, component] - 1.96*sigma,
                      demo_states[:, component] + 1.96*sigma,
                      color="C0", alpha=0.2, label="95% model band")
    axis.legend(fontsize="small")
    axis.grid(alpha=0.25)
demo_axes[1].set_xlabel("Time [s]")
demo_figure.tight_layout()
plt.show()


## E1: follow a particle track (25 minutes)

Ten parallel detector planes lie at $z=1,\ldots,10$ m, each 1 m tall in x.
Gaussian position resolution: $\sigma=0.01$ m. Each plane causes an independent
Gaussian slope kick of standard deviation 0.02: local Q adds $0.02^2$ to slope
variance, leaving position unchanged.

1. **Simulate tracks.** Choose starting x and slope at z = 0; for example,
   independent zero-mean Gaussians with standard deviations 0.1 m and 0.1.
   Propagate with scattering and generate noisy measurements.
2. **Write a Kalman filter**, state (x, slope), with an initial state
   and large covariance. Predict, then update at each plane. Measure before
   scattering; estimate the incoming state and covariance C at the last plane.
3. **Run 1,000 tracks.** Histogram both pulls,
   $(\hat x_i-x_i^{true})/\sqrt{C_{ii}}$. Fit or report means and standard
   deviations; are they consistent with $N(0,1)$?

Use Python, NumPy and Matplotlib; notation: state x, covariance C, transport F,
process noise Q, projection H, measurement variance V, gain K.
[Slides](https://www.nikhef.nl/~wouterh/teaching/TrackingAndVertexing/index.html).

We suggest feeding this prompt into your favourite AI.
The most beautiful illustration before noon wins a **pakje stroopwafels**.

**Completion:** explain one predict/update and keep code, seed, figure and pull statistics.

Reference: optional cell below; includes virtual hits outside the active height.


In [ ]:
# Try your implementation first, or enable the supplied reference for comparison.
run_tracking_reference = False
if run_tracking_reference:
    from day2 import tracking
    detector = tracking.Detector(scattering_slope_std=0.02)
    tracking_result = tracking.ensemble(tracks=1000, seed=7, detector=detector)
    tracking_output = next_attempt(output / "tracking")
    tracking.save(tracking_result, tracking_output)
    print(json.dumps(tracking_result["summary"], indent=2))
    display(Image(filename=str(tracking_output / "track.png")))
    display(Image(filename=str(tracking_output / "pulls.png")))


## Optional: tracking conventions and illustration

**Conventions to agree.** The reference measures first and scatters afterwards.
Its filtered state at the last plane is therefore the incoming state, before
that plane's kick. The kick covariance at a plane is
$Q_{\mathrm{plane}}=\mathrm{diag}(0,0.02^2)$. Over the next gap d,

$$F=\begin{pmatrix}1&d\\0&1\end{pmatrix},\qquad
Q_{\mathrm{gap}}=FQ_{\mathrm{plane}}F^T
=0.02^2\begin{pmatrix}d^2&d\\d&1\end{pmatrix},\qquad
H=(1,0),\qquad V=0.01^2\,\mathrm{m}^2.$$

A kick changes position uncertainty only after propagation. The first gap from
z = 0 to z = 1 contains no scattering plane. The reference starts truth with the
example 0.1 m / 0.1 spreads, and uses a zero-mean fit with initial standard
deviations 1 m / 1. A broad fit prior can affect pulls; check its influence
rather than imposing unit width by rescaling the errors.

**Optional plot challenge:** reproduce the green widening prediction bands, measured hits and dotted initial direction.

The reference below supplies a comparison or fallback. Its all-plane control includes virtual hits outside the 1 m active height and reports the number of fully contained tracks. Finite-aperture mode skips outside hits and retains all attempted tracks; acceptance and missing-hit information affect the pull interpretation.


Continue with [02 calibration](02_calibration.ipynb).
